# Hands-On with Docling for IBM watsonx
## Developer Notebook (Part 2B)

**Use case:** You are a compliance analyst at *Vantara Bank*. Your task is to process a bundle of regulatory documents (PDF, DOCX, EPUB) and extract the key metrics needed to fill a standardised Compliance Assessment Form.

**Nothing runs on your machine:** documents are converted by the **Docling for IBM watsonx** managed service, and every model (embeddings and LLM) is served by **watsonx.ai**.

**This notebook covers:**
1. Connect to the Docling for IBM watsonx managed service
2. Convert documents
3. Inspect document structure
4. Export to DocLang and query with `dclq`
5. Chunk documents for RAG
6. RAG pipeline with LangChain + Milvus + watsonx.ai
7. Information extraction with a Pydantic schema
8. Putting it all together — fill the Compliance Assessment Form
9. *(Optional)* Fill the form with Docling Agent

---
## Setup

Install the Python packages used in this notebook. They are all lightweight clients:
no ML framework (PyTorch) and no model weights are installed.

In [ ]:
# Install dependencies (safe to re-run; already installed packages are skipped)
! uv pip install -q \
    "docling-slim[service-client,feat-chunking]" \
    langchain-core langchain-ibm langchain-classic langchain-milvus "pymilvus[milvus_lite]" \
    python-dotenv dclq rich \
    mellea pyyaml typer tabulate

# For the optional Section 9: docling-agent declares the full docling package (with PyTorch) as a
# dependency, but the editing agent only needs docling-core. --no-deps keeps the install small.
! uv pip install -q --no-deps docling-agent

Your credentials live in the `.env` file at the root of the lab folder (`lab-1499/.env`),
which you prepared in Part 1. The watsonx.ai values are shared by the class: the instructors put
them in `~/lab-1499-watsonx.txt` on your VM, and the bootstrap copied them into `.env`. The next
cell finds the lab folder wherever Jupyter was started from, loads `.env` (and that file, for any
watsonx.ai value `.env` is missing), and checks that all the required values are present.

In [ ]:
import logging
import os
import warnings
from pathlib import Path

from dotenv import dotenv_values, load_dotenv

# Keep the output readable: silence library chatter that does not affect the lab
logging.basicConfig(level=logging.ERROR)
logging.getLogger("grpc._server").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["GRPC_VERBOSITY"] = "ERROR"


def find_lab_dir(start: Path) -> Path:
    # The lab folder is the first parent that holds both .env.example and docs/
    for folder in [start, *start.parents]:
        if (folder / ".env.example").is_file() and (folder / "docs").is_dir():
            return folder
    raise FileNotFoundError(f"Could not find the lab-1499 folder above {start}")


LAB_DIR = find_lab_dir(Path.cwd().resolve())
DOCS_DIR = LAB_DIR / "docs"
OUTPUT_DIR = LAB_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

env_file = LAB_DIR / ".env"
if not env_file.is_file():
    raise FileNotFoundError(f"{env_file} is missing: copy .env.example to .env and fill it in (lab guide, section 3.6)")
load_dotenv(env_file)

# The watsonx.ai credentials are shared by the class: the instructors put them in
# ~/lab-1499-watsonx.txt and the bootstrap copies them into .env. If .env still
# has no real value for one, take it from that file.
wx_file = Path.home() / "lab-1499-watsonx.txt"
if wx_file.is_file():
    for name, value in dotenv_values(wx_file).items():
        current = os.environ.get(name, "")
        if name.startswith("WX_") and value and (not current or current.startswith("your-")):
            os.environ[name] = value

REQUIRED = ["DOCLING_SERVICE_URL", "DOCLING_SERVICE_API_KEY", "WX_API_KEY", "WX_PROJECT_ID"]
missing = [name for name in REQUIRED if os.environ.get(name, "").startswith("your-") or not os.environ.get(name)]
if missing:
    raise RuntimeError(
        f"Set these variables in {env_file}: {', '.join(missing)} "
        f"(the WX_ ones are shared by the class, in {wx_file})"
    )

DOCLING_SERVICE_URL = os.environ["DOCLING_SERVICE_URL"]
DOCLING_SERVICE_API_KEY = os.environ["DOCLING_SERVICE_API_KEY"]
WX_API_KEY = os.environ["WX_API_KEY"]
WX_PROJECT_ID = os.environ["WX_PROJECT_ID"]
WX_URL = os.environ.get("WX_URL") or "https://us-south.ml.cloud.ibm.com"

print("Lab folder:     ", LAB_DIR)
print("Docling service:", DOCLING_SERVICE_URL[:40] + "...")
print("watsonx.ai:     ", WX_URL)

---
## Section 1 — Connect to the Managed Service

`DoclingServiceClient` is the Python entry point for **Docling for IBM watsonx**.
Its call shape mirrors the local `DocumentConverter` — swap the URL and API key and
your code is portable between local and managed execution.

In [ ]:
from docling.service_client import DoclingServiceClient

client = DoclingServiceClient(
    url=DOCLING_SERVICE_URL,
    api_key=DOCLING_SERVICE_API_KEY,
)
print("Client ready.")

---
## Section 2 — Convert Documents

We convert the three documents that make up Vantara Bank's Q3 compliance bundle.
All heavy work (OCR, table structure detection, layout analysis) happens on the
managed service — **no local ML models or GPU required**.

In [ ]:
# Document sources — local paths or public URLs
SOURCES = [
    DOCS_DIR / "vantara_risk_disclosure.pdf",
    DOCS_DIR / "q3_audit_summary.docx",
    DOCS_DIR / "regulatory_guidelines_2026.epub",
]

In [ ]:
# Convert one document
result = client.convert(source=SOURCES[0])
doc = result.document

print(f"Document: {doc.name}")
print(f"Status:   {result.status}")
print("\n--- Markdown preview (first 1000 chars) ---\n")
print(doc.export_to_markdown()[:1000])

`convert_all()` sends several documents to the service at once and yields each result as soon as it
is ready. It drives its own event loop, which cannot run inside Jupyter's, so in a notebook we call it
from a worker thread. In a regular Python script you can call it directly.

In [ ]:
# Convert all three documents concurrently
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=1) as pool:
    all_results = pool.submit(
        lambda: list(client.convert_all(source=SOURCES, max_concurrency=3))
    ).result()

docs = {}
for r in all_results:
    name = r.input.file.name if r.input and r.input.file else str(r.input)
    print(f"  {name:45s}  {r.status.value}")
    if r.document:
        docs[name] = r.document

print(f"\nSuccessfully converted: {len(docs)}/{len(SOURCES)} documents")

---
## Section 3 — Inspect Document Structure

A `DoclingDocument` is a rich tree: headings nest sections, which contain
paragraphs, tables, figures, lists, and more. You can iterate the tree in Python
to understand the document without sending it to an LLM.

In [ ]:
# Print the heading outline of the first document
from docling_core.types.doc import SectionHeaderItem, TitleItem

primary_doc = docs.get("vantara_risk_disclosure.pdf") or list(docs.values())[0]

print("=== Heading outline ===")
for item, _ in primary_doc.iterate_items():
    if isinstance(item, TitleItem):
        print(f"# {item.text}")
    elif isinstance(item, SectionHeaderItem):
        print(f"{'  ' * item.level}{'#' * (item.level + 1)} {item.text}")

In [ ]:
# Inspect tables
print(f"Tables found: {len(primary_doc.tables)}")
for i, table in enumerate(primary_doc.tables):
    print(f"\n--- Table {i + 1} ---")
    print(table.export_to_markdown(doc=primary_doc))

In [ ]:
# Inspect pictures / figures
print(f"Pictures found: {len(primary_doc.pictures)}")
for pic in primary_doc.pictures:
    caption = pic.caption_text(primary_doc)
    if caption:
        print(f"  Caption: {caption[:120]}")

---
## Section 4 — Export to DocLang and Query with `dclq`

**DocLang** (`.dclg` / `.dclx`) is an open XML-based ISO standard for structured
documents. Every heading, paragraph, table cell, and footnote has an XPath address.

`dclq` lets you grep, list, outline, and XPath-query DocLang files — **no LLM needed**.

In [ ]:
# Export every converted document to a DocLang archive (.dclx, a ZIP bundle)
dclx_paths = {}
for name, d in docs.items():
    dclx_paths[name] = OUTPUT_DIR / f"{Path(name).stem}.dclx"
    d.save_as_doclang_archive(dclx_paths[name])
    print(f"Saved DocLang file: {dclx_paths[name].relative_to(LAB_DIR)}")

dclx_path = dclx_paths["vantara_risk_disclosure.pdf"]

In [ ]:
# Inspect the document structure
! dclq inspect {dclx_path}

In [ ]:
# Print the heading outline with XPath addresses
! dclq outline {dclx_path}

In [ ]:
# Search for risk-related content
! dclq grep -i 'risk exposure' {dclx_path}

In [ ]:
# Retrieve a specific section by XPath (replace the XPath with one from the outline above)
# Example: dclq show doc.dclx '/heading[5]' --section
! dclq show {dclx_path} '/heading[2]' --section --max-chars 800

In [ ]:
# List all table cells on page 1
! dclq list {dclx_path} --type table_cell --page 1

> 📖 `dclq` is experimental. Full command reference: https://github.com/docling-project/docling-core/tree/main/packages/dclq

---
## Section 5 — Chunk Documents for RAG

The `DoclingDocument` returned by the service already knows its structure, so chunking is a fast,
deterministic step that runs on that structure — no model inference involved.

`HybridChunker` splits by the heading hierarchy first, then merges or splits the pieces so each chunk
fits the token budget of the embedding model. It only needs that model's **tokenizer** (a small
vocabulary file) to count tokens; the embeddings themselves are computed by watsonx.ai in Section 6.
Each chunk keeps its heading breadcrumb and its provenance (page and position in the source).

In [ ]:
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

# The embedding model served by watsonx.ai in Section 6, and its tokenizer
EMBEDDING_MODEL = "ibm/granite-embedding-278m-multilingual"
EMBEDDING_TOKENIZER = "ibm-granite/granite-embedding-278m-multilingual"

chunker = HybridChunker(
    tokenizer=HuggingFaceTokenizer.from_pretrained(model_name=EMBEDDING_TOKENIZER, max_tokens=512),
)

doc_chunks = [(name, chunk) for name, d in docs.items() for chunk in chunker.chunk(dl_doc=d)]
print(f"{len(doc_chunks)} chunks from {len(docs)} documents\n")


def chunk_pages(chunk) -> list[int]:
    return sorted({prov.page_no for item in chunk.meta.doc_items for prov in item.prov})


for name, chunk in doc_chunks[:3]:
    print(f"--- {name} · page {chunk_pages(chunk)} · {' > '.join(chunk.meta.headings or [])} ---")
    print(chunk.text[:300])
    print()

---
## Section 6 — RAG Pipeline

We build a retrieval-augmented generation pipeline using:
- the **Docling chunks** from Section 5, wrapped as LangChain documents
- **watsonx.ai** (`ibm/granite-embedding-278m-multilingual`) to embed them
- **Milvus Lite** as the vector store (a single local file)
- **watsonx.ai** (`ibm/granite-4-h-small`) as the LLM for generation

In [ ]:
# Ingestion — wrap the chunks as LangChain documents, embed them with watsonx.ai, store them in Milvus
from tempfile import mkdtemp

from langchain_core.documents import Document
from langchain_ibm import WatsonxEmbeddings
from langchain_milvus import Milvus

lc_docs = [
    Document(
        # contextualize() prefixes the chunk with its headings, which improves retrieval
        page_content=chunker.contextualize(chunk=chunk),
        metadata={
            "source": name,
            "pages": ", ".join(str(p) for p in chunk_pages(chunk)),
            "headings": " > ".join(chunk.meta.headings or []),
        },
    )
    for name, chunk in doc_chunks
]

embeddings = WatsonxEmbeddings(
    model_id=EMBEDDING_MODEL,
    url=WX_URL,
    project_id=WX_PROJECT_ID,
    apikey=WX_API_KEY,
)

milvus_uri = str(Path(mkdtemp()) / "compliance.db")
vector_store = Milvus.from_documents(
    documents=lc_docs,
    embedding=embeddings,
    collection_name="compliance_demo",
    connection_args={"uri": milvus_uri},
    index_params={"index_type": "FLAT", "metric_type": "COSINE"},
    drop_old=True,
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
print(f"Vector store ready with {len(lc_docs)} chunks.")

In [ ]:
# LLM — watsonx.ai
from langchain_ibm import ChatWatsonx

LLM_MODEL = "ibm/granite-4-h-small"

llm = ChatWatsonx(
    model_id=LLM_MODEL,
    url=WX_URL,
    project_id=WX_PROJECT_ID,
    apikey=WX_API_KEY,
    params={"temperature": 0, "max_tokens": 2048},
)

In [ ]:
# RAG pipeline
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

PROMPT_TEMPLATE = """You are a compliance analyst assistant.
Answer the question using only the information in the provided context.
If the answer is not in the context, say "Not found in the documents".

Context:
{context}

Question: {input}
Answer:"""

prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)
qa_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)


def ask(question: str) -> str:
    resp = rag_chain.invoke({"input": question})
    print(f"Q: {question}")
    print(f"A: {resp['answer']}")
    print("Sources:")
    for d in resp["context"]:
        # Only paginated formats (PDF) have page numbers
        page = f", page {d.metadata['pages']}" if d.metadata["pages"] else ""
        print(f"  · {d.metadata['source']}{page} — {d.metadata['headings']}")
    print()
    return resp["answer"]

In [ ]:
# Try it out
ask("What is the total risk exposure reported by Vantara Bank?")
ask("What are the main regulatory obligations described in these documents?")
ask("Were any material findings raised in the audit?")

> 💡 **Beyond this lab — visual grounding.** Every Docling chunk carries its provenance: the page
> number *and* the bounding box of each element it came from. With the page images returned by the
> service, a RAG application can highlight exactly where in the original PDF an answer was found.
> Building that viewer is out of scope here; see the
> [visual grounding example](https://docling-project.github.io/docling/examples/visual_grounding/)
> in the Docling documentation.

---
## Section 7 — Information Extraction

To fill the form we need **specific typed fields**, not free-text answers. We describe the form as a
Pydantic schema and ask the watsonx.ai LLM to return an instance of it with `with_structured_output`.

Docling makes this reliable: the three documents arrive as clean Markdown — headings, lists, and
tables preserved — whatever their original format, so the LLM reads them as structured text
instead of raw PDF, DOCX, or EPUB bytes.

> 🔜 **Coming soon:** Docling for IBM watsonx will support schema-based extraction as a native
> functionality of the service, so you will be able to send the schema with the documents and get the
> typed fields back directly.

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint


class ComplianceForm(BaseModel):
    """Compliance Assessment Form — Vantara Bank."""

    entity_name: str | None = Field(
        default=None,
        description="Legal name of the reporting entity",
        examples=["Vantara Bank"],
    )
    reporting_period: str | None = Field(
        default=None,
        description="Quarter and year covered by this report",
        examples=["Q3 2026"],
    )
    total_risk_exposure_usd: float | None = Field(
        default=None,
        description="Total risk exposure in USD millions",
    )
    highest_risk_category: str | None = Field(
        default=None,
        description="The single highest-ranked risk category",
        examples=["Credit Risk", "Market Risk", "Operational Risk"],
    )
    regulator_reference_number: str | None = Field(
        default=None,
        description="Regulatory filing or reference number",
        examples=["FRB-2026-0847"],
    )
    key_obligations: list[str] = Field(
        default_factory=list,
        description="List of key regulatory obligations mentioned",
    )
    material_findings: list[str] = Field(
        default_factory=list,
        description="Material findings raised in audit or review",
    )
    sign_off_date: str | None = Field(
        default=None,
        description="Date the audit report was signed off (YYYY-MM-DD)",
        examples=["2026-10-15"],
    )


print("Schema defined.")

In [ ]:
# Give the LLM all three converted documents at once, each tagged with its file name
bundle = "\n\n".join(
    f'<document name="{name}">\n{d.export_to_markdown()}\n</document>' for name, d in docs.items()
)
print(f"Sending {len(bundle):,} characters of Markdown from {len(docs)} documents")

extraction_llm = llm.with_structured_output(ComplianceForm)

form = extraction_llm.invoke(
    [
        (
            "system",
            "You are a compliance analyst. Fill in the Compliance Assessment Form using only facts "
            "stated in the documents. Leave a field empty when the documents do not state it.",
        ),
        ("human", bundle),
    ]
)

merged_form = form.model_dump()
print("\n=== Extracted Compliance Form ===")
rprint(merged_form)

---
## Section 8 — Putting It All Together

We now combine the extracted fields with an LLM-generated narrative summary
and render the completed Compliance Assessment Form as an HTML report.

In [ ]:
# Generate a narrative compliance summary using watsonx.ai
import json

summary_prompt = f"""You are a senior compliance analyst at Vantara Bank.
Based on the following extracted compliance data, write a concise 3-sentence executive
summary of the Q3 compliance posture. Focus on total risk exposure, the highest risk
category, and the most critical regulatory obligations.

Extracted data:
{json.dumps(merged_form, indent=2)}

Executive summary:"""

summary_response = llm.invoke(summary_prompt)
compliance_summary = summary_response.content
print(compliance_summary)

In [ ]:
# Render the completed form as an HTML report
from IPython.display import HTML, display


def render_compliance_form(form: dict, summary: str) -> str:
    rows = ""
    field_labels = {
        "entity_name": "Entity Name",
        "reporting_period": "Reporting Period",
        "total_risk_exposure_usd": "Total Risk Exposure (USD M)",
        "highest_risk_category": "Highest Risk Category",
        "regulator_reference_number": "Regulatory Reference No.",
        "key_obligations": "Key Obligations",
        "material_findings": "Material Findings",
        "sign_off_date": "Sign-off Date",
    }
    for field, label in field_labels.items():
        value = form.get(field)
        if isinstance(value, list):
            value_html = "<ul>" + "".join(f"<li>{v}</li>" for v in value) + "</ul>" if value else "<em>Not found</em>"
        elif value is None:
            value_html = "<em style='color:#999'>Not found</em>"
        else:
            value_html = str(value)
        rows += f"<tr><td style='font-weight:600;padding:6px 12px;'>{label}</td><td style='padding:6px 12px'>{value_html}</td></tr>"

    return f"""
<div style="font-family:system-ui,sans-serif;max-width:720px;margin:0 auto">
  <h2 style="border-bottom:2px solid #0062ff;padding-bottom:8px">Compliance Assessment Form</h2>
  <p style="color:#555"><em>Vantara Bank — auto-populated by Docling for IBM watsonx</em></p>
  <table style="border-collapse:collapse;width:100%;margin:16px 0">
    <thead><tr style="background:#f0f0f0">
      <th style="padding:6px 12px;text-align:left">Field</th>
      <th style="padding:6px 12px;text-align:left">Value</th>
    </tr></thead>
    <tbody>{rows}</tbody>
  </table>
  <h3>Executive Summary</h3>
  <blockquote style="border-left:4px solid #0062ff;padding:8px 16px;background:#f5f9ff;margin:0">
    {summary}
  </blockquote>
</div>
"""

html_report = render_compliance_form(merged_form, compliance_summary)
display(HTML(html_report))

In [ ]:
# Save the report to disk
report_path = OUTPUT_DIR / "compliance_assessment_form_filled.html"
report_path.write_text(
    f"<!DOCTYPE html><html><head><meta charset='utf-8'><title>Compliance Assessment</title></head>"
    f"<body>{html_report}</body></html>",
    encoding="utf-8",
)
print(f"Report saved to: {report_path.relative_to(LAB_DIR)}")

---
## Section 9 — (Optional) Fill the Form with Docling Agent

[`docling-agent`](https://github.com/docling-project/docling-agent) provides agents that read and edit
a `DoclingDocument` from natural-language instructions. Here the `DoclingEditingAgent` takes an empty
Compliance Assessment Form — itself a `DoclingDocument` — and fills its table with the values
extracted in Section 7.

The agent talks to its LLM through a small *backend* interface. It ships backends for local servers
(Ollama, LM Studio, a LiteLLM proxy), so we write a tiny one that sends its requests straight to
**watsonx.ai** through the same `ChatWatsonx` client used above.

In [ ]:
# A docling-agent backend that runs every request on watsonx.ai
import sys
import types

# docling-agent imports its extraction agent eagerly, and that one needs the full local docling
# stack (PyTorch). We only use the editing agent, so we register an empty placeholder for it.
sys.modules.setdefault("docling.document_extractor", types.ModuleType("docling.document_extractor"))
sys.modules["docling.document_extractor"].DocumentExtractor = None

from docling_agent.backends.base import BaseBackend, BaseSession
from docling_agent.task_model import BackendConfig, ModelConfig
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

logging.getLogger("docling_agent").setLevel(logging.WARNING)


class WatsonxSession(BaseSession):
    """One conversation with a watsonx.ai chat model."""

    def __init__(self, chat: ChatWatsonx, system_prompt: str | None = None):
        self._chat = chat
        self._messages = [SystemMessage(system_prompt)] if system_prompt else []

    def instruct(self, prompt, *, requirements=None, retry_budget=1) -> str:
        # The agent validates the reply itself; we spell its requirements out in the prompt
        if requirements:
            prompt += "\n\nRequirements:\n" + "\n".join(f"- {r.description}" for r in requirements)
        self._messages.append(HumanMessage(prompt))
        reply = self._chat.invoke(self._messages).content
        self._messages.append(AIMessage(reply))
        return reply


class WatsonxBackend(BaseBackend):
    """Creates watsonx.ai sessions for the models named in the backend config."""

    backend_type = "watsonx"

    def __init__(self, config: BackendConfig):
        self.config = config

    @classmethod
    def from_config(cls, config: BackendConfig) -> "WatsonxBackend":
        return cls(config)

    def create_session(self, *, model: str, system_prompt: str | None = None) -> BaseSession:
        chat = ChatWatsonx(
            model_id=model,
            url=WX_URL,
            project_id=WX_PROJECT_ID,
            apikey=WX_API_KEY,
            params={"temperature": 0, "max_tokens": 2048},
        )
        return WatsonxSession(chat, system_prompt)


backend = WatsonxBackend(
    BackendConfig(models=ModelConfig(reasoning=LLM_MODEL, writing=LLM_MODEL))
)
print("watsonx.ai backend ready.")

In [ ]:
# Build the empty form as a DoclingDocument, from the fields of compliance_assessment_form.json
from docling_core.types.doc import DocItemLabel, DoclingDocument, TableCell, TableData

template = json.loads((DOCS_DIR / "compliance_assessment_form.json").read_text())
form_fields = [key for key in template if not key.startswith("_")]


def cell(text: str, row: int, col: int) -> TableCell:
    return TableCell(
        text=text,
        start_row_offset_idx=row,
        end_row_offset_idx=row + 1,
        start_col_offset_idx=col,
        end_col_offset_idx=col + 1,
        column_header=row == 0,
    )


cells = [cell("Field", 0, 0), cell("Value", 0, 1)]
for row, field in enumerate(form_fields, start=1):
    cells += [cell(field, row, 0), cell("", row, 1)]

form_doc = DoclingDocument(name="compliance_assessment_form")
form_doc.add_title(text="Compliance Assessment Form — Vantara Bank")
form_doc.add_heading(text="Assessment fields", level=1)
form_doc.add_text(label=DocItemLabel.TEXT, text="Populate every value from the Q3 2026 compliance bundle.")
caption = form_doc.add_text(label=DocItemLabel.CAPTION, text="Compliance Assessment Form fields and values")
form_table = form_doc.add_table(
    data=TableData(num_rows=len(form_fields) + 1, num_cols=2, table_cells=cells),
    caption=caption,
)

print(form_doc.export_to_markdown())

# Save the empty form now: the agent edits the document in place
empty_form_path = OUTPUT_DIR / "compliance_assessment_form_empty.dclx"
form_doc.save_as_doclang_archive(empty_form_path)
print(f"Saved the empty form to {empty_form_path.relative_to(LAB_DIR)}")

> 🔍 Open **<https://doclang.ai/viewer/>** and drag `output/compliance_assessment_form_empty.dclx` onto it.
> The form you just built in Python is a regular DocLang document, with the same title, heading,
> caption, and table nodes as a converted PDF.

In [ ]:
# Let the agent fill the form table with the extracted values
from docling_agent.agents import DoclingEditingAgent

agent = DoclingEditingAgent(backend=backend, tools=[])

# The agent plans its edit from the document outline, which lists headings but not tables,
# so we name the table by its reference (#/tables/0) in the task.
filled_doc = agent.run(
    task=(
        f"Update the content of the table {form_table.self_ref}: fill in its Value column with these "
        "values. Write list values as a semicolon-separated list:\n" + json.dumps(merged_form, indent=2)
    ),
    document=form_doc,
)

print(filled_doc.export_to_markdown())

In [ ]:
# Save the agent-filled form, as DocLang and as HTML
filled_form_path = OUTPUT_DIR / "compliance_assessment_form_agent_filled.dclx"
filled_doc.save_as_doclang_archive(filled_form_path)
print(f"Saved to {filled_form_path.relative_to(LAB_DIR)}")

agent_path = OUTPUT_DIR / "compliance_assessment_agent_filled.html"
filled_doc.save_as_html(agent_path)
print(f"Saved to {agent_path.relative_to(LAB_DIR)}")

> 🔍 Drag `output/compliance_assessment_form_agent_filled.dclx` onto **<https://doclang.ai/viewer/>**
> and compare it with the empty form: the document structure is unchanged, and the agent wrote the
> values into the cells of the table.

---
## Summary

In this notebook you:

| Step | What you did | Component |
|------|-------------|------------------|
| 1 | Connected to the managed service | `DoclingServiceClient` |
| 2 | Converted PDF, DOCX, and EPUB concurrently | `client.convert_all()` |
| 3 | Explored document structure in Python | `doc.iterate_items()`, `doc.tables` |
| 4 | Exported to DocLang and ran XPath queries | `save_as_doclang_archive()`, `dclq` |
| 5 | Chunked the documents for RAG | `HybridChunker` |
| 6 | Built a RAG pipeline with LangChain + Milvus | `WatsonxEmbeddings`, `ChatWatsonx` |
| 7 | Extracted typed fields across all documents | Pydantic + `with_structured_output` |
| 8 | Generated an HTML compliance report | LLM summary + HTML render |
| 9 | *(Optional)* Filled the form with an agent | `DoclingEditingAgent` on watsonx.ai |

**Next steps:**
- Try the IBM Bob track (Part 2A) to do the same workflow conversationally
- Explore the [Docling documentation](https://docling-project.github.io/docling/) for advanced pipeline options
- Check the [Docling Agent examples](https://github.com/docling-project/docling-agent/tree/main/examples) for chunkless RAG, document writing, and batch enrichment